# Übersetzungs-Test RM ⇄ DE (Colab)

Modell: [ZurichNLP/romansh-nllb-1.3b-ct2](https://huggingface.co/ZurichNLP/romansh-nllb-1.3b-ct2) · Lizenz CC-BY-NC-4.0 (nicht kommerziell)

**Nur mit unkritischem Testtext verwenden.** Der Text wird auf einem Google-Server verarbeitet, nicht auf dem eigenen Rechner.

**Ablauf**
1. Menü *Laufzeit → Laufzeittyp ändern → T4-GPU* wählen (viel schneller; ohne GPU geht es auch, dauert aber länger).
2. *Laufzeit → Alle ausführen*. Beim ersten Mal dauert Zelle 1–2 ein paar Minuten (Installation, Modell-Download ca. 1.3 GB).
3. In Zelle 3 den Text zwischen die drei Anführungszeichen einfügen (eine Frase pro Zeile, z. B. aus dem Subtitle Studio), Richtung und Idiom wählen und die Zelle mit ▶ ausführen.
4. Ergebnis kopieren und im Subtitle Studio in Schritt 3 einfügen.
**Direkt mit dem Subtitle Studio:** Nur die Zellen **1 und 4** ausführen (2 und 3 überspringen) und auf den Link tippen. Der Subtitle Studio übersetzt dann über dieses Colab (nur unkritischer Text).
Bei «CUDA out of memory»: *Laufzeit → Sitzung neu starten* und von vorn beginnen.


In [ ]:
#@title 1 · Installieren
!pip -q install ctranslate2 sentencepiece huggingface_hub transformers nvidia-cublas-cu12


In [ ]:
#@title 2 · Modell laden
# Speicher freigeben, falls die Zelle schon einmal lief (sonst: «CUDA out of memory»)
import os, gc
os.system("pkill -f UTstudio-colab-server.py; pkill cloudflared")
try: del tr
except NameError: pass
gc.collect()
import ctranslate2, time
from huggingface_hub import snapshot_download
from transformers import NllbTokenizer
t0 = time.time()
path = snapshot_download("ZurichNLP/romansh-nllb-1.3b-ct2")
tok = NllbTokenizer.from_pretrained(path)
dev = "cuda" if ctranslate2.get_cuda_device_count() > 0 else "cpu"
tr = ctranslate2.Translator(path, device=dev)
print(f"Modell bereit auf {dev.upper()} ({time.time()-t0:.0f} s)")

def translate(src, tgt, lines, bs=16):
    tok.src_lang = src
    out = []
    for i in range(0, len(lines), bs):
        part = lines[i:i+bs]
        batch = [tok.convert_ids_to_tokens(tok.encode(t)) for t in part]
        res = tr.translate_batch(batch, target_prefix=[[tgt]]*len(batch), beam_size=4, max_decoding_length=256)
        out += [tok.decode(tok.convert_tokens_to_ids(r.hypotheses[0][1:]), skip_special_tokens=True) for r in res]
    return out

import ctypes, glob, site
def preload_cublas():
    for sp in site.getsitepackages():
        for pat in ("nvidia/cublas/lib/libcublasLt.so.12*", "nvidia/cublas/lib/libcublas.so.12*"):
            for f in glob.glob(f"{sp}/{pat}"):
                try: ctypes.CDLL(f, mode=ctypes.RTLD_GLOBAL)
                except OSError: pass
preload_cublas()
try:
    translate("deu_Latn", "rm-rumgr", ["Test."]); print("GPU funktioniert ✓")
except Exception as e:
    print("GPU nicht nutzbar, wechsle auf CPU (langsamer):", str(e)[:120])
    tr = ctranslate2.Translator(path, device="cpu"); print("CPU bereit")


In [ ]:
# 3 · Übersetzen – Richtung und Idiom wählen, Text unten zwischen die """ einfügen (eine Frase pro Zeile), dann ▶ drücken
RICHTUNG = "RM → DE" #@param ["RM → DE", "DE → RM"]
IDIOM = "Rumantsch Grischun" #@param ["Rumantsch Grischun", "Sursilvan", "Sutsilvan", "Surmiran", "Puter", "Vallader"]
TEXT = """
Buna saira, chars spectaturs.
Oz discurrin nus davart la neiv.
"""

import html
from IPython.display import HTML, display
IDIOMS = {"Rumantsch Grischun":"rm-rumgr","Sursilvan":"rm-sursilv","Sutsilvan":"rm-sutsilv","Surmiran":"rm-surmiran","Puter":"rm-puter","Vallader":"rm-vallader"}
rm = IDIOMS[IDIOM]
src, tgt = (rm, "deu_Latn") if RICHTUNG == "RM → DE" else ("deu_Latn", rm)
lines = [l.strip() for l in TEXT.strip().splitlines()]
idx = [i for i, l in enumerate(lines) if l]          # leere Zeilen bleiben leer (Zeilenzahl stimmt)
res = translate(src, tgt, [lines[i] for i in idx])
full = [""] * len(lines)
for i, r in zip(idx, res): full[i] = r.replace("\n", " ").strip()
result = "\n".join(full)
print(f"✓ {len(idx)} Zeilen übersetzt ({RICHTUNG}, {IDIOM}) – bitte prüfen:\n")
print(result)
display(HTML('<textarea id="res" style="width:100%;height:200px;font:14px sans-serif">' + html.escape(result) + '</textarea><br>'
  '<button onclick="const t=document.getElementById(\'res\');t.select();document.execCommand(\'copy\');this.textContent=\'Kopiert ✓\'">Ergebnis kopieren</button>'))


In [ ]:
#@title 4 · Mit dem Subtitle Studio verbinden (optional)
# Startet das Modell als Server und macht ihn über einen Tunnel für den Subtitle Studio erreichbar.
# Danach auf den Link unten tippen – der Subtitle Studio übersetzt dann direkt über dieses Colab (Zelle 3 wird nicht gebraucht).
# ACHTUNG: Der Text läuft dann über Cloudflare (Tunnel) und Google (Colab) – nur unkritischer Testtext!
import subprocess, secrets, re, time, os, glob, site, urllib.request
from IPython.display import HTML, display
os.system("pkill -f UTstudio-colab-server.py; pkill cloudflared")  # alten Server beenden (sonst GPU-Speicher doppelt belegt)
try:
    del tr; import gc; gc.collect()   # Modell aus Zelle 2 freigeben: der Server lädt sein eigenes
except NameError: pass
PORT = 8787
token = secrets.token_urlsafe(16)
urllib.request.urlretrieve("https://raw.githubusercontent.com/dombuzz/tools/main/lab/UTstudio-colab-server.py", "UTstudio-colab-server.py")
urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "cloudflared")
os.chmod("cloudflared", 0o755)
env = dict(os.environ)
libs = [d for sp in site.getsitepackages() for d in glob.glob(f"{sp}/nvidia/cublas/lib")]
env["LD_LIBRARY_PATH"] = ":".join(libs + [env.get("LD_LIBRARY_PATH", "")])
srv = subprocess.Popen(["python3", "UTstudio-colab-server.py", "--token", token, "--port", str(PORT)], env=env, stdout=open("server.log", "w"), stderr=subprocess.STDOUT)
print("Lade Modell im Server …")
for _ in range(300):
    log = open("server.log").read()
    if "Bereit auf" in log: break
    if srv.poll() is not None: print(log); raise SystemExit("Server wurde beendet")
    time.sleep(1)
else: raise SystemExit("Server nicht bereit:\n" + log[-500:])
tun = subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"], stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(90):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if m: url = m.group(0); break
    time.sleep(1)
if not url: raise SystemExit("Tunnel nicht gestartet:\n" + open("tunnel.log").read()[-500:])
link = f"https://tools.dombuzz.ch/lab/UTstudio-colab#server={url}&token={token}"
print("Bereit. Server-Log:", log.strip().splitlines()[-1])
display(HTML(f'<p><a href="{link}" target="_blank" style="font-size:18px">▶ Subtitle Studio (Colab) öffnen (verbunden mit diesem Colab)</a></p>'
             '<p>Das Notebook-Fenster offen lassen. Beenden: <i>Laufzeit → Sitzung beenden</i>.</p>'))
